# Setup

Install dependencies, clone the repo, and import required libraries and modules.

In [ ]:
# Install dependencies
!pip -q install transformers sentence-transformers tqdm pandas numpy pillow


In [ ]:
# Clone the project repo
!git clone https://github.com/gizayceylan/FakeNews.git

# Add it to Python path
import sys
sys.path.append("/content/FakeNews")


In [ ]:
# Imports
import os, json
import numpy as np
import pandas as pd
import torch

from tqdm import tqdm
from PIL import Image
from google.colab import files

from transformers import CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer, util

# Data

Load the pre-processed and balanced Fakeddit subset.

In [ ]:
# Unzip fakeddit_images.zip from /content/FakeNews/assets/content/master_pipeline_assets/
!unzip -q /content/FakeNews/assets/fakeddit_images.zip -d /content/FakeNews/assets

In [ ]:
# Load the pre-processed metadata
subset_path = "/content/FakeNews/assets/fakeddit_balanced_subset.csv"
balanced_df = pd.read_csv(subset_path)

# Define the path for the image directory (which was unzipped)
img_dir = "/content/FakeNews/assets/content/fakeddit_images"

# Update the DataFrame paths to reflect the new location
balanced_df['image_path'] = balanced_df['image_path'].apply(
    lambda p: os.path.join(img_dir, os.path.basename(p))
)

print(f"Loaded {len(balanced_df)} clean samples.")
print("Shape subset:", balanced_df.shape)
balanced_df.head()


In [ ]:
# Display the number of fake/real labels
print(balanced_df["label"].value_counts())

In [ ]:
# Test a sample
Image.open(balanced_df["image_path"].iloc[0])


# Models

In [ ]:
# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# CLIP
clip_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_name)
clip_processor = CLIPProcessor.from_pretrained(clip_name)
clip_model.to(device)
clip_model.eval()
print("\n-----------------------------------------------------")
print("CLIP loaded successfully.")
print("-----------------------------------------------------")

# SentenceTransformer
sim_model = SentenceTransformer("all-mpnet-base-v2")
sim_model.to(device)
print("\n-----------------------------------------------------")
print("MPNet loaded successfully.")
print("-----------------------------------------------------")


# Similarity

In [ ]:
def context_alignment(user_text: str, AI_caption: str, image_path: str) -> dict:
    """
    Computes two-way alignment scores:
    1. Semantic: Text vs Caption
    2. Cross-modal: Image vs Text (User Claim Alignment)
    """
    # 1. Semantic Similarity (MPNet - Text-to-Text)
    emb1 = sim_model.encode(user_text, convert_to_tensor=True, device=device)
    emb2 = sim_model.encode(AI_caption, convert_to_tensor=True, device=device)
    st_sim = util.cos_sim(emb1, emb2).item()

    # 2. Cross-Modal Similarity (CLIP - Image-to-Text)
    image = Image.open(image_path).convert("RGB")

    # Get separate image & text features
    image_inputs = clip_processor(images=image, return_tensors="pt").to(device)
    text_inputs = clip_processor(text=[user_text], return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        img_feats = clip_model.get_image_features(**image_inputs)
        txt_feats = clip_model.get_text_features(**text_inputs)

    # Normalize to unit length
    img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
    txt_feats = txt_feats / txt_feats.norm(dim=-1, keepdim=True)

    # Cosine similarity: dot product of normalized vectors
    img_text_sim = float((img_feats @ txt_feats.T).squeeze().item())  # in [-1, 1]

    return {
        "text_to_caption_similarity": st_sim,
        "image_to_text_similarity": img_text_sim
    }

# Calibration

In [ ]:
# Calibrate on 150 samples (prepped and balanced)
sub = balanced_df.reset_index(drop=True)

t2c_scores, i2t_scores = [], []

for _, row in tqdm(sub.iterrows(), total=len(sub), desc="Computing similarity bins"):
    out = context_alignment(
        user_text=row["clean_title"],
        AI_caption=row["blip2_caption"],
        image_path=row["image_path"]
    )
    t2c_scores.append(float(out["text_to_caption_similarity"]))
    i2t_scores.append(float(out["image_to_text_similarity"]))

t25, t50, t75 = np.quantile(t2c_scores, [0.25, 0.50, 0.75]).tolist()
i25, i50, i75 = np.quantile(i2t_scores, [0.25, 0.50, 0.75]).tolist()

sim_bins = {"t2c": [t25, t50, t75], "i2t": [i25, i50, i75]}

print("\nsim_bins =", sim_bins)


# Save

Save the calibrated similarity bins.

In [ ]:
# Save the final JSON
output_file = "fakeddit_calibrated_sim_bins.json"
with open(output_file, "w") as f:
    json.dump(sim_bins, f, indent=2)

# Download the JSON
files.download(output_file)
